# modular-diffusion: a line of flow-matching fine-tunes beside Stable Diffusion 1.5

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/modular-diffusion/blob/v0.17.0/colab/lune_line_tables.ipynb)

Stable Diffusion 1.5 predicts the noise on 1,000 discrete steps. A line of fine-tunes of its U-Net (the same layout, every tensor of the same name and shape) was retrained by flow matching: each predicts the velocity `v = noise - x0` of `x_s = (1 - s) x0 + s noise` on a continuous time `s`, its U-Net fed `1000 s`, sampled by Euler steps on a grid warped by a shift. This notebook reads those sources beside SD1.5 with no training at all, on a small card (the U-Nets in half precision, the image decoder in float32). The stretches of the noise range keep their edges (0.35 and 0.75 with a half-width of 0.06) on each source's own coordinate `s`.

PART 1, the instruments (set `PART = 1`): THE LATENT SCALE of each flow source (its unguided samples' standard deviation over the rows' own clean latents; inside 0.9 to 1.1 the standard scale is kept, otherwise the ratio is applied to the latents given to the source and taken off its pictures), with SD1.5's own samples as the control; A GUIDANCE LADDER (guidance 1, 2, 4, 6 and 7.5 on 256 prompts, the text-to-picture score and the kernel distance to the rows' own pictures, a contact sheet of 16 a rung; the guidance is chosen by the rule written in the second cell); THE SAMPLER on this card against a second implementation of the same formulas; THE SHIFT (each source at the other shift).

PART 2, the tables (set `PART = 2` and fill `KAPPA` and `GUIDANCE` from part 1's report): SD1.5's table of single removals again as the control; for the two main flow sources the born read of every single removal and of the named shapes by stretch, and the prompt's weight by stretch; the distance of the weights group by group for every pair; THE EXCHANGE TABLE, each source with ONE named group of its U-Net (`conv_in`, `time_embedding`, `down_blocks.0` to `.3`, `mid_block`, `up_blocks.0` to `.3`, `conv_norm_out + conv_out`) replaced by another source's, read against its own answer in each stretch, for every ordered pair, with its controls (nothing exchanged, everything exchanged, a random donor) and then the two groups that read largest split into their resnets and attentions; last a band-trained child against its parent, by stretch and on ten slices of `s`.

The cells: (1) install at a release tag; (2) the part, the card, the sources fetched and checked (their revisions and the stored SHA-256 of every file recorded; the cell stops if one is missing or of the wrong size), the probe rows, the job list, a smoke run of every job of the part on tiny stand-ins, a timed smoke on the real sources and the projected minutes; (3) the job list a phase at a time (a phase's jobs are made from the finished results of the phases before it by the rules written in the second cell, never by hand): each finished job shipped at once, the report shipped, and the machine given back only when everything finished was shipped.

RESUME: run the cells again. Finished jobs keep their markers under `OUT`; on a new machine they come back from the Hub, so nothing runs twice.

In [ ]:
# C1 - install at a tag (extras on the name); torch is not a dependency, so the runtime's torch stays
%pip install -q "modular-diffusion[capture,hub,data,guard,judge] @ git+https://github.com/AbstractEyes/modular-diffusion@v0.17.0"
import os, sys, time, json, gc, hashlib, shutil, statistics, threading
import numpy as np
import torch, geolip.modular_diffusion as MD
assert tuple(int(x) for x in MD.__version__.split(".")) >= (0, 17, 0), "RESTART REQUIRED: Runtime > Restart session, then run C1 again"
import diffusers, transformers, huggingface_hub
print("torch", torch.__version__, "| diffusers", diffusers.__version__, "| transformers", transformers.__version__,
      "| huggingface_hub", huggingface_hub.__version__, "| package", MD.__version__)
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print("card:", props.name, f"| memory {props.total_memory / 2**30:.1f} GiB | cpus: {os.cpu_count()}")
else:
    print("card: NONE (choose a GPU runtime)")
OUT = "/content/lune_line_tables"       # the output root; keep it the same to resume
os.makedirs(OUT, exist_ok=True)

`PART` at the top chooses the part; part 2 stops until `KAPPA` holds the four flow sources' latent-scale ratios and `GUIDANCE` the two main flow sources' guidance from part 1's report (the band pair is read at its parent's ratio). The rules that make a later phase run only on a complete set (every rung of a ladder, every ordered pair's exchange): an incomplete set stops the run with the reason. The card is printed. SD1.5 comes at its pinned revision; each flow source's U-Net folder alone (`<path>/unet/*`) comes at its repository's current revision, which is recorded with the stored SHA-256 of every file; each weights file (SD1.5's too) must be 3,438,167,536 bytes and equal to its stored SHA-256, and a weights file with no stored SHA-256 stops the cell. The probe rows are the first 256 pictures of COCO val2017 with their content hash checked, encoded once by SD1.5's image encoder. The job list is a list of phases: each phase's jobs are made by a function of its inputs, and the inputs of a later phase (the latent scale, the guidance, the two groups to split) come from the finished results by the rules written here. The smoke runs every phase on tiny random stand-ins written here (tiny U-Net folders of other seeds) and checks the exchange controls, the sampler check and the copies' reads; it is shipped. Then a few rows of the scale read, one group of the exchange and a picture job of eight rows run on the real sources as the roster runs a job; their seconds give the projection.

In [ ]:
# C2 - the part, the card, the sources, the probe rows, the job list, a smoke, the projection
PART = 1          # 1: the instruments (latent scale, guidance ladder, sampler check, shift); 2: the tables
KAPPA = {}        # PART 2: {"main": ..., "flux": ..., "parent": ..., "child": ...}, by the scale rule of part 1's report
GUIDANCE = {}     # PART 2: {"main": ..., "flux": ...}, each source's guidance by the ladder rule of part 1's report
assert PART in (1, 2), "PART is 1 or 2"
if PART == 2:
    assert set(KAPPA) == {"main", "flux", "parent", "child"} and set(GUIDANCE) == {"main", "flux"}, (
        "part 2 runs with KAPPA and GUIDANCE filled from part 1's report (its scale rule and its ladder rule): fill both "
        "dictionaries at the top of this cell, then run it again")
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
import subprocess
import pyarrow.parquet as pq
from geolip.modular_diffusion import hub, run
from geolip.modular_diffusion.capture.data import center_crop, encode_shards, parquet_pictures, read_index, synthetic_shards
from geolip.modular_diffusion.capture.teacher import _perturb, load_teacher, tiny_unet
from geolip.modular_diffusion.experiments import band_needs, capture, lineage
TEACHER, PROBE, CLIP, RUNS, UNETS = f"{OUT}/sd15", f"{OUT}/probe", f"{OUT}/clip", f"{OUT}/runs", f"{OUT}/unets"
REPO = "AbstractPhil/geolip-multirotary-diffusion-collective"   # the results repository ("" = nothing shipped)
PREFIX = "modular_diffusion/catalog/lune_line"
NOT_SHIPPED = ("SHIPPED", "FROM_HUB")                            # two markers; the rest of a job folder ships
UNET_BYTES = 3_438_167_536                                       # every U-Net weights file of this measurement
DEVICE = "cuda"
os.makedirs(RUNS, exist_ok=True)

# the card: inference only, the U-Nets in half precision (the image decoder in float32): a small card is enough
assert torch.cuda.is_available(), "no card: choose a GPU runtime (a T4 is enough)"
CARD = torch.cuda.get_device_properties(0)
CARD_GIB = CARD.total_memory / 2**30
print(f"card: {CARD.name} | memory {CARD_GIB:.1f} GiB")

# the sources: Stable Diffusion 1.5 (noise prediction on steps 0 .. 999) and four flow-matching fine-tunes of its U-Net
# (the velocity v = noise - x0 on a continuous time s, the U-Net fed 1000 s, zero states as the unguided side), each
# read with the shift it was trained at; the text encoder, tokenizer and image decoder of every source are SD1.5's
SD15 = ("stable-diffusion-v1-5/stable-diffusion-v1-5", "451f4fe16113bff5a5d2269ed5ad43b0592e9a14")
SOURCES = {
    "main": dict(repo="AbstractPhil/sd15-flow-lune", path="checkpoint-00052296", shift=2.0),
    "flux": dict(repo="AbstractPhil/sd15-flow-lune-flux", path="flux_t2_6_pose_t4_6_port_t1_4/checkpoint-00018765", shift=2.5),
    "parent": dict(repo="AbstractPhil/sd15-flow-lune", path="pretrain_pose_controlnet_t500_700/checkpoint-00008312", shift=2.5),
    "child": dict(repo="AbstractPhil/sd15-flow-lune", path="pretrain_pose_controlnet_v7_v10_t400_600/checkpoint-00007968",
                  shift=2.5),
}
NEEDED = ("main", "flux", "parent", "child")          # both parts: part 1 reads the scale of all four

# the fetch: SD1.5 at its pinned revision; each flow source's U-Net folder alone (<path>/unet/*) at its repository's
# revision of this moment, recorded with each file's stored SHA-256; every file must come back, every U-Net weights file
# must be 3,438,167,536 bytes and equal to its stored SHA-256, else the cell stops here (no other file is read)
snapshot_download(repo_id=SD15[0], revision=SD15[1],
                  allow_patterns=["model_index.json", "scheduler/scheduler_config.json", "tokenizer/*", "text_encoder/config.json",
                                  "text_encoder/model.safetensors", "unet/config.json", "unet/diffusion_pytorch_model.safetensors",
                                  "vae/config.json", "vae/diffusion_pytorch_model.safetensors"], local_dir=TEACHER)


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 24), b""):
            h.update(chunk)
    return h.hexdigest()


def stored_sha256(f):
    """The SHA-256 the Hub stores for a file (its large-file record), or None."""
    lfs = getattr(f, "lfs", None)
    return getattr(lfs, "sha256", None) or (lfs.get("sha256") if isinstance(lfs, dict) else None)


def check_weights(name, path, f, problems):
    """A weights file's record; a missing file, a size other than UNET_BYTES, no stored SHA-256 or another SHA-256 is
    a problem that stops the cell."""
    entry = {"bytes": getattr(f, "size", None) if f is not None else None,
             "stored_sha256": stored_sha256(f) if f is not None else None, "present": os.path.exists(path)}
    if not entry["present"]:
        problems.append(f"{name}: no unet/diffusion_pytorch_model.safetensors")
        return entry
    entry["local_bytes"] = os.path.getsize(path)
    entry["local_sha256"] = sha256_of(path)
    if entry["local_bytes"] != UNET_BYTES:
        problems.append(f"{name}: the U-Net weights are {entry['local_bytes']:,} bytes, not {UNET_BYTES:,}")
    elif not entry["stored_sha256"]:
        problems.append(f"{name}: the Hub stores no SHA-256 for its U-Net weights")
    elif entry["local_sha256"] != entry["stored_sha256"]:
        problems.append(f"{name}: the U-Net weights' SHA-256 is not the stored one")
    return entry


def fetch_unets(needed, api=None):
    """{name: folder}, {name: record}, [problems]: each U-Net folder fetched and checked (SD1.5's U-Net too)."""
    api = api or HfApi()
    folders, records, problems = {"sd15": f"{TEACHER}/unet"}, {}, []
    try:
        info = api.model_info(SD15[0], revision=SD15[1], files_metadata=True)
        f = next((x for x in info.siblings if x.rfilename == "unet/diffusion_pytorch_model.safetensors"), None)
    except Exception as e:
        problems.append(f"sd15: {repr(e)[:200]}")
        f = None
    records["sd15"] = {"repo": SD15[0], "revision": SD15[1], "files": {"unet/diffusion_pytorch_model.safetensors": check_weights(
        "sd15", f"{TEACHER}/unet/diffusion_pytorch_model.safetensors", f, problems)}}
    for name in needed:
        s = SOURCES[name]
        try:
            info = api.model_info(s["repo"], files_metadata=True)
            files = {f.rfilename: f for f in info.siblings if f.rfilename.startswith(f"{s['path']}/unet/")}
            got = snapshot_download(repo_id=s["repo"], revision=info.sha, allow_patterns=[f"{s['path']}/unet/*"],
                                    local_dir=f"{UNETS}/{name}")
        except Exception as e:
            problems.append(f"{name}: {repr(e)[:200]}")
            continue
        folder = f"{got}/{s['path']}/unet"
        rec = {"repo": s["repo"], "revision": info.sha, "path": s["path"], "shift": s["shift"], "files": {}}
        weights_rel = f"{s['path']}/unet/diffusion_pytorch_model.safetensors"
        for rel, f in sorted(files.items()):
            local = f"{got}/{rel}"
            if rel == weights_rel:
                rec["files"][rel] = check_weights(name, local, f, problems)
            else:          # a small file (its configuration): kept in the Hub's git, no large-file SHA-256 exists
                rec["files"][rel] = {"bytes": getattr(f, "size", None), "stored_sha256": stored_sha256(f),
                                     "present": os.path.exists(local)}
        if weights_rel not in files:
            rec["files"][weights_rel] = check_weights(name, f"{got}/{weights_rel}", None, problems)
        if not os.path.exists(f"{folder}/config.json"):
            problems.append(f"{name}: no unet/config.json")
        folders[name], records[name] = folder, rec
    return folders, records, problems


UNET, FETCHED, PROBLEMS = fetch_unets(NEEDED)
assert not PROBLEMS, "a source did not come back as it should: " + "; ".join(PROBLEMS)
for name, rec in FETCHED.items():
    print(f"{name}: {rec['repo']} at {rec['revision']}" + (f", {rec['path']}" if "path" in rec else ""))

# the probe rows: the first 256 pictures of COCO val2017's first parquet file at a pinned revision with their first
# captions (the probe of the earlier band measurement, its content hash checked), encoded once by SD1.5's image encoder;
# the first 64 are the tables' probe, all 256 are the picture reads' prompts and rows
PARQUET = hf_hub_download(repo_id="lmms-lab-encoder/COCO-Caption2017", repo_type="dataset",
                          revision="3bdd5827e243cc3084ac69a1111e69c3ab9193ff", filename="data/val-00000-of-00002.parquet",
                          local_dir=f"{OUT}/coco")
N_PROBE = 256
EXPECTED_CONTENT_HASH = "93d333a65f7bbdac6ced9143ff55852273c4e2a64deb7f8f641aeb1878f35c44"


def content_hash(pictures_, captions, size=512):
    h = hashlib.sha256()
    for p, c in zip(pictures_, captions):
        a = ((center_crop(p, size) + 1.0) * 127.5).round().clamp(0, 255).to(torch.uint8)    # the array the encoder sees
        h.update(a.numpy().tobytes())
        h.update(c.encode("utf-8") + b"\0")
    return h.hexdigest()


if not (os.path.exists(f"{PROBE}/index.json") and sum(read_index(PROBE)["counts"]) == N_PROBE):
    pictures_, captions, meta = parquet_pictures(PARQUET, N_PROBE)
    HASH = content_hash(pictures_, captions)
    print("probe content hash", HASH, "| expected", EXPECTED_CONTENT_HASH)
    assert HASH == EXPECTED_CONTENT_HASH, "the probe pictures are not the expected data"
    with_vae = load_teacher(TEACHER, device=DEVICE, dtype=torch.float32, vae=True)
    encode_shards(pictures_, captions, PROBE, with_vae.vae, with_vae.tokenize, shard_size=64, batch=16, size=512, device=DEVICE)
    json.dump({"content_hash": HASH, "pictures": [{"row": i, "id": m.get("id"), "license": m.get("license")} for i, m in enumerate(meta)]},
              open(f"{PROBE}/pictures.json", "w", encoding="utf-8"), indent=1)
    del with_vae, pictures_
    gc.collect()
    torch.cuda.empty_cache()
print("probe shards:", read_index(PROBE)["counts"])
# the text-to-picture judge at a pinned revision: its settings and weights only
snapshot_download(repo_id="openai/clip-vit-large-patch14", revision="32bd64288804d66eefd0ccbe215aa642df71cc41",
                  allow_patterns=["config.json", "model.safetensors", "preprocessor_config.json"], local_dir=CLIP)

# the job list: each phase's jobs are (tag, "experiment arguments", the jobs it waits for, kind, what it is), made from
# its inputs; a phase's inputs come from the finished results of the phases before it by the rules below
SIZES = dict(rows=256, group=16, blocks=10, probe_rows=64, per_band=4, prompts=128, sd15_steps=50, flow_steps=30,
             per_slice=4, sampler_rows=4)
LADDER = (1.0, 2.0, 4.0, 6.0, 7.5)                   # guidance 1 is unguided
SCALE_GUIDANCE = 4.0                                 # the scale read's guided samples of a flow source
CTX = dict(runs=RUNS, unet=UNET, teacher=TEACHER, probe=PROBE, clip=CLIP, **SIZES)
COMMON = "device=cuda threads=4"
PAIRS = (("sd15", "main"), ("main", "sd15"), ("main", "flux"), ("flux", "main"), ("sd15", "flux"), ("flux", "sd15"))
DISTANCES = (("sd15", "main"), ("main", "flux"), ("sd15", "flux"))
RANDOM_GROUPS = ("mid_block", "up_blocks.3")         # the random donor's two groups (reference: main)
FLOW = ("main", "flux", "parent", "child")
g_ = lambda v: f"{float(v):g}"                      # noqa: E731
k_ = lambda v: f"{float(v):.4g}"                    # noqa: E731


def src(ctx, name, kappa=1.0):
    """The source's arguments: SD1.5, or a flow source's U-Net folder, kind, shift and latent-scale ratio."""
    if name == "sd15":
        return f"teacher={ctx['teacher']}"
    s = SOURCES[name]
    return f"teacher={ctx['teacher']} unet={ctx['unet'][name]} kind=flow shift={s['shift']} kappa={kappa}"


def pic(ctx, name, kappa, guidance, shift=None):
    s = src(ctx, name, kappa)
    if shift is not None:
        s = s.replace(f" shift={SOURCES[name]['shift']} ", f" shift={shift} ")
    steps = ctx["sd15_steps"] if name == "sd15" else ctx["flow_steps"]
    return (f"capture variant=pictures {s} dtype=fp16 part_dtype=fp16 parts=clean=teacher,middle=teacher,noisy=teacher "
            f"pic_data={ctx['probe']} pic_first=0 pic_n={ctx['rows']} group={ctx['group']} noise_seed=2000 "
            f"sample_steps={steps} guidance={guidance} judge=inception clip={ctx['clip']} sheet_n=16")


def compare(ctx, tag, a, b, meta):
    return (tag, f"capture variant=picture_compare a={ctx['runs']}/{a} b={ctx['runs']}/{b} paired=1 blocks={ctx['blocks']}",
            [a, b], "compare", meta)


def probe_args(ctx):
    return (f"probe_data={ctx['probe']} probe_first=0 probe_rows={ctx['probe_rows']} probe_seed=1000 "
            f"probe_per_band={ctx['per_band']} chunk=16")


ROWS_TAG = lambda ctx: f"p1_rows_n{ctx['rows']}"     # noqa: E731


def phase_scale(ctx):
    """PART 1, first: the rows' own pictures, the scale read of each source, the sampler check on the card."""
    jobs = [(ROWS_TAG(ctx), f"capture variant=pictures source=rows {src(ctx, 'sd15')} dtype=fp16 pic_data={ctx['probe']} "
             f"pic_first=0 pic_n={ctx['rows']} judge=inception clip={ctx['clip']} sheet_n=16", [], "pictures",
             {"read": "rows"})]
    jobs.append((f"p1_scale_sd15_ddim{ctx['sd15_steps']}_g7.5_n{ctx['rows']}",
                 f"lineage variant=scale {src(ctx, 'sd15')} dtype=fp16 pic_data={ctx['probe']} pic_n={ctx['rows']} "
                 f"group={ctx['group']} noise_seed=2000 sample_steps={ctx['sd15_steps']} guidance=7.5", [], "scale",
                 {"read": "scale", "source": "sd15", "guidance": 7.5}))
    for name in FLOW:
        sh = SOURCES[name]["shift"]
        jobs.append((f"p1_scale_{name}_s{g_(sh)}_steps{ctx['flow_steps']}_g{g_(SCALE_GUIDANCE)}_n{ctx['rows']}",
                     f"lineage variant=scale {src(ctx, name)} dtype=fp16 pic_data={ctx['probe']} pic_n={ctx['rows']} "
                     f"group={ctx['group']} noise_seed=2000 sample_steps={ctx['flow_steps']} guidance={SCALE_GUIDANCE}",
                     [], "scale", {"read": "scale", "source": name, "guidance": SCALE_GUIDANCE, "shift": sh}))
    jobs.append((f"p1_sampler_flux_s2.5_steps{ctx['flow_steps']}_g4",
                 f"lineage variant=sampler_check {src(ctx, 'flux')} dtype=fp32 pic_data={ctx['probe']} "
                 f"pic_n={ctx['sampler_rows']} sample_steps={ctx['flow_steps']} guidance=4.0", [], "check",
                 {"read": "sampler", "source": "flux"}))
    return jobs


def rule_kappa(ctx, done):
    """The scale rule (registered): each flow source's kappa from its finished scale job (lineage.scale_rule); only on
    the complete set (all four flow sources), else the later phases are not made."""
    out = {}
    for t, meta in done.items():
        if meta.get("read") == "scale" and meta["source"] != "sd15":
            out[meta["source"]] = json.load(open(f"{ctx['runs']}/{t}/result.json", encoding="utf-8"))["kappa_to_use"]
    assert set(out) == set(FLOW), f"the scale read did not finish for every flow source: {sorted(out)} of {list(FLOW)}"
    return out


def phase_ladder(ctx, kappa):
    """PART 1, second: the guidance ladder of each flow source, each rung against the rows' own pictures."""
    jobs = []
    for name in ("main", "flux"):
        sh = SOURCES[name]["shift"]
        for g in LADDER:
            t = f"p1_ladder_{name}_s{g_(sh)}_k{k_(kappa[name])}_g{g_(g)}_n{ctx['rows']}"
            meta = {"read": "ladder", "source": name, "guidance": g, "shift": sh, "kappa": kappa[name]}
            jobs.append((t, pic(ctx, name, kappa[name], g), [], "pictures", meta))
            jobs.append(compare(ctx, f"p1_cmp_ladder_{name}_s{g_(sh)}_g{g_(g)}_vs_rows", t, ROWS_TAG(ctx), dict(meta, read="ladder_cmp")))
    return jobs


def ladder_pick(rungs):
    """The guidance rule (registered): among the rungs whose kernel distance to the rows' own pictures is not worse than
    the best rung's by more than twice the best rung's standard error, the highest mean text-to-picture score; a tie goes
    to the lower rung. rungs: [(guidance, clip mean, kid block mean, kid standard error)]."""
    best = min(rungs, key=lambda r: (r[2], r[0]))
    allowed = [r for r in rungs if r[2] <= best[2] + 2 * best[3]]
    top = max(r[1] for r in allowed)
    return min(r[0] for r in allowed if r[1] == top)


def ladder_rungs(ctx, done, name):
    out = []
    for t, meta in done.items():
        if meta.get("read") == "ladder_cmp" and meta["source"] == name:
            r = json.load(open(f"{ctx['runs']}/{t}/result.json", encoding="utf-8"))
            out.append((meta["guidance"], r["clip_a"], r["kid"]["block_mean"], r["kid"]["block_std"] / r["kid"]["blocks"] ** 0.5))
    return sorted(out)


def rule_guidance(ctx, done):
    """The guidance rule on the complete ladder only: every rung of LADDER of each source finished, else no pick (the
    later phase is not made and the cell says why)."""
    out = {}
    for name in ("main", "flux"):
        rungs = ladder_rungs(ctx, done, name)
        got = sorted(r[0] for r in rungs)
        assert got == sorted(LADDER), (f"the ladder of {name} is not complete (rungs {got} of {list(LADDER)}): the guidance "
                                       "rule runs only on all five rungs")
        out[name] = ladder_pick(rungs)
    return out


def phase_shift(ctx, kappa, guidance):
    """PART 1, last: each flow source at the other shift, at its ladder's guidance; against the rows' own pictures and,
    row by row, against its own rung at the trained shift."""
    jobs = []
    for name, other in (("main", 2.5), ("flux", 2.0)):
        sh = SOURCES[name]["shift"]
        t = f"p1_shift_{name}_s{g_(other)}_k{k_(kappa[name])}_g{g_(guidance[name])}_n{ctx['rows']}"
        meta = {"read": "shift", "source": name, "guidance": guidance[name], "shift": other, "kappa": kappa[name]}
        jobs.append((t, pic(ctx, name, kappa[name], guidance[name], shift=other), [], "pictures", meta))
        trained = f"p1_ladder_{name}_s{g_(sh)}_k{k_(kappa[name])}_g{g_(guidance[name])}_n{ctx['rows']}"
        jobs.append(compare(ctx, f"p1_cmp_shift_{name}_s{g_(other)}_vs_rows", t, ROWS_TAG(ctx), dict(meta, read="shift_cmp_rows")))
        jobs.append(compare(ctx, f"p1_cmp_shift_{name}_s{g_(other)}_vs_s{g_(sh)}", t, trained, dict(meta, read="shift_cmp_trained")))
    return jobs


def phase_tables(ctx, kappa, guidance):
    """PART 2, first: SD1.5's table A again (the control), tables A and B of each flow source, the distances, the
    exchanges of every ordered pair with a random donor in one group."""
    kap = dict(kappa, sd15=1.0)
    bn = (f"data={ctx['probe']} seeds=0,1 rows={ctx['probe_rows']} per_band={ctx['per_band']} prompts={ctx['prompts']} "
          "group=32 chunk=16 dtype=fp32")
    jobs = [("p2_tableA_sd15_fp32", f"band_needs {src(ctx, 'sd15')} {bn} do=r1,r2", [], "band",
             {"read": "tableA", "source": "sd15"})]
    for name in ("main", "flux"):
        sh = SOURCES[name]["shift"]
        jobs.append((f"p2_tablesAB_{name}_s{g_(sh)}_k{k_(kappa[name])}_g{g_(guidance[name])}",
                     f"band_needs {src(ctx, name, kappa[name])} {bn} steps={ctx['flow_steps']} guidance={guidance[name]} "
                     "sample_dtype=fp16 do=r1,r2,r3", [], "band",
                     {"read": "tablesAB", "source": name, "kappa": kappa[name], "guidance": guidance[name]}))
    for a, b in DISTANCES:
        jobs.append((f"p2_distance_{a}_to_{b}", f"lineage variant=distance {src(ctx, a)} donor={ctx['unet'][b]}", [],
                     "distance", {"read": "distance", "reference": a, "donor": b}))
    for ref, donor in PAIRS:
        jobs.append((f"p2_exchange_{ref}_from_{donor}", f"lineage variant=group_exchange {src(ctx, ref, kap[ref])} "
                     f"donor={ctx['unet'][donor]} dtype=fp16 {probe_args(ctx)}", [], "exchange",
                     {"read": "exchange", "reference": ref, "donor": donor}))
    for g in RANDOM_GROUPS:
        jobs.append((f"p2_exchange_main_from_random_{g}", f"lineage variant=group_exchange {src(ctx, 'main', kap['main'])} "
                     f"donor=random donor_seed=1 groups={g} dtype=fp16 {probe_args(ctx)}", [], "exchange",
                     {"read": "exchange_random", "reference": "main", "donor": "random", "group": g}))
    return jobs


def rule_split(ctx, done):
    """The split rule (registered here): of the groups that hold resnets or attentions, the two whose largest read over
    every exchange of a real pair and every stretch is the largest (a tie: the earlier group in the U-Net): the same two
    for every pair. Only on the complete set (all six ordered pairs finished), else no pick."""
    have = {(m["reference"], m["donor"]) for m in done.values() if m.get("read") == "exchange"}
    assert have == set(PAIRS), (f"the exchanges of {sorted(set(PAIRS) - have)} did not finish: the split rule runs only on "
                                "all six ordered pairs")
    best = {g: None for g in lineage.SPLITTABLE}
    for t, meta in done.items():
        if meta.get("read") == "exchange":
            table = json.load(open(f"{ctx['runs']}/{t}/result.json", encoding="utf-8"))["table"]
            for g in lineage.SPLITTABLE:
                v = max(x for x in table[g].values() if x is not None)
                best[g] = v if best[g] is None else max(best[g], v)
    ranked = sorted((g for g in best if best[g] is not None), key=lambda g: (-best[g], lineage.GROUPS.index(g)))
    assert len(ranked) >= 2, "the exchanges did not finish"
    return ranked[:2]


def phase_split(ctx, kappa, groups):
    kap = dict(kappa, sd15=1.0)
    jobs = []
    for ref, donor in PAIRS:
        jobs.append((f"p2_split_{ref}_from_{donor}_{groups[0]}_{groups[1]}", f"lineage variant=group_exchange "
                     f"{src(ctx, ref, kap[ref])} donor={ctx['unet'][donor]} split={','.join(groups)} dtype=fp16 {probe_args(ctx)}",
                     [], "exchange", {"read": "split", "reference": ref, "donor": donor, "groups": list(groups)}))
    return jobs


def phase_band_pair(ctx, kappa):
    """PART 2, last: the band pair (the child read against its parent by stretch and on ten slices of s; the distance each
    group moved; the child's groups exchanged into the parent), at the PARENT's latent scale."""
    k = kappa["parent"]
    return [(f"p2_child_parent_k{k_(k)}", f"lineage variant=child_parent {src(ctx, 'child', k)} parent={ctx['unet']['parent']} "
             f"slices=10 per_slice={ctx['per_slice']} dtype=fp16 {probe_args(ctx)}", [], "pair",
             {"read": "child_parent", "reference": "parent", "donor": "child", "kappa": k}),
            ("p2_distance_parent_to_child", f"lineage variant=distance {src(ctx, 'parent')} donor={ctx['unet']['child']}", [],
             "distance", {"read": "distance", "reference": "parent", "donor": "child"}),
            (f"p2_exchange_parent_from_child_k{k_(k)}", f"lineage variant=group_exchange {src(ctx, 'parent', k)} "
             f"donor={ctx['unet']['child']} dtype=fp16 {probe_args(ctx)}", [], "exchange",
             {"read": "exchange_pair", "reference": "parent", "donor": "child", "kappa": k})]


def phases(part):
    """[(phase name, make(ctx, done) -> jobs)]: done = {tag: meta} of the finished jobs of the phases before."""
    if part == 1:
        return [("scale", lambda ctx, done: phase_scale(ctx)),
                ("ladder", lambda ctx, done: phase_ladder(ctx, rule_kappa(ctx, done))),
                ("shift", lambda ctx, done: phase_shift(ctx, rule_kappa(ctx, done), rule_guidance(ctx, done)))]
    return [("tables", lambda ctx, done: phase_tables(ctx, KAPPA, GUIDANCE)),
            ("split", lambda ctx, done: phase_split(ctx, KAPPA, rule_split(ctx, done))),
            ("band_pair", lambda ctx, done: phase_band_pair(ctx, KAPPA))]


def placeholder_jobs(ctx, part):
    """Every phase's jobs with stand-in inputs (kappa 1, guidance 4, two groups), for the job's size before it runs."""
    one, four = {n: 1.0 for n in FLOW}, {"main": 4.0, "flux": 4.0}
    if part == 1:
        return phase_scale(ctx) + phase_ladder(ctx, one) + phase_shift(ctx, one, four)
    return (phase_tables(ctx, KAPPA or one, GUIDANCE or four) + phase_split(ctx, KAPPA or one, ["mid_block", "up_blocks.1"])
            + phase_band_pair(ctx, KAPPA or one))


# the smoke on tiny random stand-ins: every phase of this part, through the same rules, with tiny U-Net folders written
# here for the sources (other seeds each) and random rows; the controls that apply are checked
SMOKE = f"{OUT}/smoke_lineage"
shutil.rmtree(SMOKE, ignore_errors=True)
synthetic_shards(f"{SMOKE}/probe", rows=12, shard_size=5, size=16, seed=4)
SMOKE_UNET = {}
for i, name in enumerate(("sd15",) + tuple(SOURCES)):      # sd15's stand-in: the tiny teacher's own U-Net (seed 0)
    seed = 0 if name == "sd15" else 10 + i
    u = tiny_unet(seed)
    _perturb(u, seed)
    u.save_pretrained(f"{SMOKE}/unets/{name}", safe_serialization=True)
    SMOKE_UNET[name] = f"{SMOKE}/unets/{name}"
SMOKE_CTX = dict(CTX, runs=f"{SMOKE}/runs", unet=SMOKE_UNET, probe=f"{SMOKE}/probe", rows=10, group=2, blocks=2,
                 probe_rows=4, per_band=2, prompts=4, sd15_steps=4, flow_steps=4, per_slice=2, sampler_rows=2)
TINY_EXTRA = {"capture": " tiny=1 judge=fake clip=fake sheet_n=4 dtype=fp32 part_dtype=", "lineage": " tiny=1 dtype=fp32",
              "band_needs": " tiny=1 dtype=fp32 sample_dtype=fp32 group=2 chunk=2"}
SMOKE_KAPPA = dict(KAPPA or {"main": 0.2, "flux": 1.0, "parent": 0.25, "child": 1.0})
SMOKE_GUIDANCE = dict(GUIDANCE or {"main": 4.0, "flux": 6.0})


def smoke_phases(part):
    if part == 1:
        return phases(1)
    return [("tables", lambda ctx, done: phase_tables(ctx, SMOKE_KAPPA, SMOKE_GUIDANCE)),
            ("split", lambda ctx, done: phase_split(ctx, SMOKE_KAPPA, rule_split(ctx, done))),
            ("band_pair", lambda ctx, done: phase_band_pair(ctx, SMOKE_KAPPA))]


S, DONE_SMOKE = {}, {}
for phase, make in smoke_phases(PART):
    for t, cmd, needs, kind, meta in make(SMOKE_CTX, DONE_SMOKE):
        exp, args = cmd.split(" ", 1)
        S[t] = run.main(exp, f"{args} {COMMON}{TINY_EXTRA[exp]}".split() + [f"out={SMOKE_CTX['runs']}/{t}"])
        DONE_SMOKE[t] = meta
for t, meta in DONE_SMOKE.items():
    r = S[t]
    if meta.get("read") in ("exchange", "exchange_random", "split", "exchange_pair"):
        assert all(r["controls"].values()), f"{t}: an exchange control failed: {r['controls']}"
    if meta.get("read") == "sampler":
        assert r["first_step_equal_to_rounding"] and r["repeat_equal"], f"{t}: the sampler check failed"
    if meta.get("read") in ("tableA", "tablesAB"):
        assert all(v == 0.0 for s in r["controls"] for v in r["controls"][s]["full"].values()), f"{t}: a copy does not read 0"
print(f"smoke: {len(S)} jobs of part {PART} on tiny stand-ins, phase by phase; the exchange controls, the sampler check "
      "and the copies' reads hold")

if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")          # placed in this process's environment only
    assert hub.ship(SMOKE, REPO, f"{PREFIX}/_smoke_part{PART}", f"the lineage tables, part {PART}: smoke run",
                    ignore=NOT_SHIPPED), "the smoke folder was not shipped: check the token and the repository before the long run"

# the real smoke on this card, each as the roster runs a job (a new process), timed: a few rows through the scale read of
# a flow source, one group of the exchange, a picture job of a few rows on a flow source (in part 2 also the band
# measurement's own timing); the projection in minutes from their seconds
JOB_ENV = dict(os.environ, OMP_NUM_THREADS="4", MKL_NUM_THREADS="4", HF_TOKEN="", PYTHONUTF8="1", PYTHONUNBUFFERED="1")
REAL_EXTRA = {"capture": "", "lineage": "", "band_needs": ""}


def as_a_job(cmd, out):
    exp, args = cmd.split(" ", 1)
    shutil.rmtree(out, ignore_errors=True)
    os.makedirs(out)
    t = time.time()
    with open(f"{out}/run.log", "wb") as fh:
        p = subprocess.run([sys.executable, "-u", "-m", "geolip.modular_diffusion.run", exp] + args.split()
                           + REAL_EXTRA[exp].split() + [f"out={out}"], env=JOB_ENV, stdout=fh, stderr=subprocess.STDOUT,
                           stdin=subprocess.DEVNULL)
    wall = time.time() - t
    tail = open(f"{out}/run.log", encoding="utf-8", errors="replace").read()[-2000:]
    assert p.returncode == 0, f"the smoke job in {out} failed:\n{tail}"
    name = "timing.json" if "variant=timing" in args else "result.json"
    return json.load(open(f"{out}/{name}", encoding="utf-8")), wall


SR = 8                                               # the real smoke's rows
FS = SIZES["flow_steps"]
R_SCALE, W_SCALE = as_a_job(f"lineage variant=scale {src(CTX, 'main')} dtype=fp16 pic_data={PROBE} pic_n={SR} group={SR} "
                            f"noise_seed=2000 sample_steps={FS} guidance={SCALE_GUIDANCE} {COMMON}", f"{OUT}/real_smoke/scale")
FWD = (R_SCALE["samples"]["unguided"]["seconds"] + R_SCALE["samples"]["guided"]["seconds"]) / (SR * FS * 3)
R_EX, W_EX = as_a_job(f"lineage variant=group_exchange {src(CTX, 'main')} donor={UNET['flux']} groups=mid_block dtype=fp16 "
                      f"probe_data={PROBE} probe_first=0 probe_rows={SR} probe_seed=1000 probe_per_band=4 chunk=16 {COMMON}",
                      f"{OUT}/real_smoke/exchange")
assert all(R_EX["controls"].values()), f"the exchange's controls on the real sources: {R_EX['controls']}"
R_PIC, W_PIC = as_a_job(pic(dict(CTX, rows=SR, group=SR), "main", 1.0, 4.0) + f" {COMMON}", f"{OUT}/real_smoke/pictures")
PIC_WORK = R_PIC["seconds_sampling"] + R_PIC["seconds_decoding"] + R_PIC["seconds_judging"]
STARTUP_S = max(0.0, W_PIC - PIC_WORK)
EX_READ_S = max(0.0, W_EX - STARTUP_S) / 7           # its reads: the answers of the model and the donor, three units, the last
PIC_ROW_S = (R_PIC["seconds_decoding"] + R_PIC["seconds_judging"]) / SR
BN = None
if PART == 2:
    BN, _ = as_a_job(f"band_needs variant=timing {src(CTX, 'main')} data={PROBE} seeds=0,1 rows={SIZES['probe_rows']} "
                     f"per_band={SIZES['per_band']} prompts={SIZES['prompts']} group=32 chunk=16 steps={FS} "
                     f"guidance=4.0 dtype=fp32 sample_dtype=fp16 {COMMON}", f"{OUT}/real_smoke/band_timing")
print(f"on this card: a row through a U-Net forward {FWD * 1000:.1f} ms in half precision; a job's start-up {STARTUP_S:.0f} s; "
      f"an exchange read of {SR} rows {EX_READ_S:.1f} s; a picture row's decoding and judging {PIC_ROW_S:.2f} s"
      + (f"; the band measurement {BN['minutes']['total']:.1f} min a source" if BN else ""))


def cost(cmd, kind):
    """A job's projected seconds from the measured rates (each job: its start-up and its forwards)."""
    a = dict(x.split("=", 1) for x in cmd.split() if "=" in x)
    if kind == "compare":
        return 15.0
    if kind == "pictures":
        n = int(a["pic_n"])
        if a.get("source") == "rows":
            return STARTUP_S + n * PIC_ROW_S
        steps, g = int(a["sample_steps"]), float(a["guidance"])
        return STARTUP_S + n * (steps * (1 if g == 1.0 else 2) * FWD + PIC_ROW_S)
    if kind == "scale":
        return STARTUP_S + int(a["pic_n"]) * int(a["sample_steps"]) * 3 * FWD
    if kind == "check":
        return STARTUP_S + 3 * int(a["pic_n"]) * int(a["sample_steps"]) * 2 * FWD * 3      # float32: about 3 times
    if kind == "distance":
        return STARTUP_S + 90.0
    if kind == "band":
        m = BN["minutes"] if BN else {"reads": 0.0, "r3": 0.0}
        return STARTUP_S + 60 * (m["reads"] + (m["r3"] if "r3" in a.get("do", "") else 0.0))
    steps = 3 * SIZES["per_band"]
    rows = int(a.get("probe_rows", SIZES["probe_rows"]))
    if kind == "pair":
        return STARTUP_S + (steps + 10 * int(a.get("per_slice", 4))) * rows * 2 * FWD
    units = 14 if "split" not in a else 8
    return STARTUP_S + (units + 3) * steps * rows * FWD


PLACEHOLDER = placeholder_jobs(CTX, PART)
COST = {t: cost(cmd, kind) for t, cmd, _, kind, _ in PLACEHOLDER}
print(f"part {PART}: {len(PLACEHOLDER)} jobs in {len(phases(PART))} phases, about {sum(COST.values()) / 60:.0f} projected minutes "
      f"on this card with one runner (" + ", ".join(f"{k} {sum(COST[t] for t, *_, kk, _ in PLACEHOLDER if kk == k) / 60:.0f}"
                                                   for k in sorted({x[3] for x in PLACEHOLDER})) + ")")
INSTRUMENTS = {"part": PART, "card": CARD.name, "card_gib": CARD_GIB, "fetched": FETCHED, "kappa": KAPPA, "guidance": GUIDANCE,
               "row_forward_seconds_fp16": FWD, "job_startup_seconds": STARTUP_S, "exchange_read_seconds_8_rows": EX_READ_S,
               "picture_row_decode_judge_seconds": PIC_ROW_S, "band_timing": BN, "projected_minutes": sum(COST.values()) / 60,
               "real_smoke_controls": R_EX["controls"]}
gc.collect()
torch.cuda.empty_cache()

One runner works each phase's roster in order. Each finished job folder is shipped at once to `REPO` under `modular_diffusion/catalog/lune_line/<job>/`, with the token from the notebook's secret `HF_TOKEN`. When a phase ends, the next phase's jobs are made from the finished results (if a rule's inputs did not finish, the later phases are not run and the report says so). At the end the summary, the job index with every job's command and what it reads, the instrument numbers and the rosters' logs are shipped to `_report_part1` or `_report_part2`. The machine is given back (`RELEASE`) only when every job of the part finished and was shipped with the report; when an upload failed, a job failed or a rule stopped the run, the machine is kept and the cell says which (run C3 again to resume).

In [ ]:
# C3 - the part's job list, a phase at a time; each finished job shipped; a progress line a minute; the summary; the machine
from geolip.modular_diffusion import jobs
RELEASE = True                     # give the machine back at the end (only when every finished job and the report were shipped)
EXTRA = {}                         # more arguments for every job of an experiment (empty in a real run)
PROGRESS_S = 60
mark = lambda t, m: os.path.exists(f"{RUNS}/{t}/{m}")
if REPO:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
INDEX = {}                         # tag -> (the command, the kind, what it is), every phase's jobs as they are made


def fetch_back(tags):
    # on a new machine: finished jobs come back with their files, so nothing runs twice
    try:
        got = snapshot_download(repo_id=REPO, allow_patterns=[f"{PREFIX}/{t}/*" for t in tags], local_dir=f"{OUT}/hub_back")
    except Exception as e:
        print("nothing fetched back from the Hub:", repr(e)[:200])
        return
    for t in tags:
        src_, dst = f"{got}/{PREFIX}/{t}", f"{RUNS}/{t}"
        if mark(t, "DONE") or not os.path.exists(f"{src_}/result.json"):
            continue
        shutil.copytree(src_, dst, dirs_exist_ok=True)
        open(f"{dst}/DONE", "w").write("fetched back from the Hub\n")
        open(f"{dst}/FROM_HUB", "w").write("fetched back from the Hub\n")


def after_job(tag, folder, ok):
    if ok and REPO:
        if hub.ship(folder, REPO, f"{PREFIX}/{tag}", f"the lineage tables, part {PART}: {tag}", ignore=NOT_SHIPPED):
            open(f"{folder}/SHIPPED", "w").write("shipped to the Hub\n")      # kept across sessions on this machine
        else:
            print(f"{tag}: NOT SHIPPED (its folder stays on this machine: {folder})", flush=True)


shipped_or_fetched = lambda t: mark(t, "SHIPPED") or mark(t, "FROM_HUB")
cap = lambda t, a, k: max(30, int(3 * cost(a, k) / 60))


def run_phase(phase, jobs_):
    tags = [t for t, *_ in jobs_]
    assert len(set(tags)) == len(tags), f"phase {phase}: two jobs of one name"
    roster = f"{OUT}/roster_part{PART}_{phase}.txt"
    with open(roster, "w", encoding="utf-8", newline="\n") as fh:
        for t, cmd, needs, kind, meta in jobs_:
            exp = cmd.split(" ", 1)[0]
            inside = [n for n in needs if n in tags]          # a job of an earlier phase is finished before this one starts
            fh.write(f"{t} | {cap(t, cmd, kind)} | geolip.modular_diffusion.run | {cmd} {COMMON}{EXTRA.get(exp, '')}"
                     + (f" | needs={','.join(inside)}" if inside else "") + "\n")
        fh.write("END\n")
    if REPO:
        fetch_back(tags)
    for t in tags:      # a claim left by an interrupted session would hold its job forever
        if mark(t, "CLAIM") and not (mark(t, "DONE") or mark(t, "FAILED")):
            os.remove(f"{RUNS}/{t}/CLAIM")
    for t in tags:      # a job that finished in an earlier session but whose upload failed: shipped again now
        if REPO and mark(t, "DONE") and not shipped_or_fetched(t):
            after_job(t, f"{RUNS}/{t}", True)
    costs = {t: cost(cmd, kind) for t, cmd, _, kind, _ in jobs_}
    t0, left0 = time.time(), sum(costs[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
    print(f"phase {phase}: {sum(mark(t, 'DONE') for t in tags)}/{len(tags)} jobs already done; about {left0 / 60:.0f} min to run")
    runner = threading.Thread(target=jobs.run_roster, daemon=True,
                              kwargs=dict(roster=roster, out=RUNS, stop=f"{OUT}/STOP", log=f"{OUT}/roster_part{PART}_{phase}.log",
                                          card_room_mib=None, ram_free_gb=4.0, max_hours=20.0, claim=True, after_job=after_job,
                                          wait_s=5.0, env={"OMP_NUM_THREADS": "4", "MKL_NUM_THREADS": "4"}))
    runner.start()
    while runner.is_alive():
        runner.join(PROGRESS_S)
        done, failed = sum(mark(t, "DONE") for t in tags), sum(mark(t, "FAILED") for t in tags)
        left = sum(costs[t] for t in tags if not (mark(t, "DONE") or mark(t, "FAILED")))
        spent = (time.time() - t0) / 60
        rate = (left0 - left) / 60 / spent if left0 > left and spent > 0 else 1.0     # projected minutes done a minute
        print(f"phase {phase}: jobs done {done}/{len(tags)} (failed {failed}); {spent:.0f} min spent; about "
              f"{left / 60 / rate:.0f} min left", flush=True)
    return [t for t in tags if mark(t, "DONE")], [t for t in tags if not mark(t, "DONE")]


DONE, NOT_DONE, STOPPED = {}, [], None
for phase, make in phases(PART):
    try:
        jobs_ = make(CTX, DONE)
    except AssertionError as e:                # a rule's inputs did not finish: the later phases are not run
        STOPPED = f"phase {phase} was not made: {e}"
        print("STOPPED:", STOPPED)
        break
    for t, cmd, needs, kind, meta in jobs_:
        INDEX[t] = {"phase": phase, "command": cmd, "kind": kind, "meta": meta}
    finished, missing = run_phase(phase, jobs_)
    DONE.update({t: INDEX[t]["meta"] for t in finished})
    NOT_DONE += missing

summary = "\n".join(m.summarize(RUNS) for m in (lineage, capture, band_needs))
print(summary)
REPORT = f"{OUT}/report_part{PART}"
os.makedirs(REPORT, exist_ok=True)
open(f"{REPORT}/summary.md", "w", encoding="utf-8").write(summary)
json.dump({"part": PART, "kappa": KAPPA, "guidance": GUIDANCE, "jobs": INDEX, "not_done": NOT_DONE, "stopped": STOPPED},
          open(f"{REPORT}/jobs.json", "w", encoding="utf-8"), indent=1)
json.dump(INSTRUMENTS, open(f"{REPORT}/instruments.json", "w", encoding="utf-8"), indent=1)
for f in os.listdir(OUT):
    if f.startswith(f"roster_part{PART}_"):
        shutil.copy(f"{OUT}/{f}", f"{REPORT}/{f.replace('.log', '_log.txt')}")
for t in INDEX:      # the logs of the jobs that failed
    if mark(t, "FAILED"):
        for f in ("run.log", "log.txt"):
            if os.path.exists(f"{RUNS}/{t}/{f}"):
                shutil.copy(f"{RUNS}/{t}/{f}", f"{REPORT}/failed_{t}_{f.replace('.', '_')}.txt")
shipped = hub.ship(REPORT, REPO, f"{PREFIX}/_report_part{PART}", f"the lineage tables, part {PART}: summary, jobs and checks") if REPO else None
unshipped = [t for t in INDEX if mark(t, "DONE") and not shipped_or_fetched(t)]
print("report shipped:", shipped, "| finished jobs not on the Hub:", ", ".join(unshipped) or "none",
      "| jobs not finished:", ", ".join(NOT_DONE) or "none", "| a rule stopped the run:", STOPPED or "no")
KEEP = []          # the machine is given back only when every job of the part finished and shipped, with the report
if not shipped:
    KEEP.append("the report was not shipped")
if unshipped:
    KEEP.append(f"these finished jobs were not shipped: {', '.join(unshipped)} (run C3 again to retry)")
if NOT_DONE:
    KEEP.append(f"these jobs failed or did not finish: {', '.join(NOT_DONE)} (their logs are in the report)")
if STOPPED:
    KEEP.append(STOPPED)
if not RELEASE:
    KEEP.append("RELEASE is False")
if not KEEP:
    from google.colab import runtime
    runtime.unassign()
else:
    print("THE MACHINE IS KEPT: " + "; ".join(KEEP))